In [1]:
# ==============================================================================
# FIGURE: Sensitivity analysis (Property 4) - feature contribution to the
# multi-dimensional outlier records of each generator (Credit dataset).
#
# Regenerated from the saved CSVs only (no model training).
# Same logic as Block A in credit_complete_pipeline.ipynb:
#   z-score of each continuous feature w.r.t. the REAL minority (train),
#   outlier record = more than 3 features simultaneously at |z| > 2.
# No in-image title: the title belongs in the LaTeX \caption.
# ==============================================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") if Path("../data").is_dir() else Path(".")   # works from scripts/ or repo root
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

CONTINUOUS_COLS = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
]

real_minority   = pd.read_csv(DATA_DIR / "real_minority_train.csv")
synthetic_ctgan = pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv")
synthetic_ctab  = pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv")

mean_real = real_minority[CONTINUOUS_COLS].astype(float).mean()
std_real  = real_minority[CONTINUOUS_COLS].astype(float).std()


def outlier_analysis(synthetic):
    z = ((synthetic[CONTINUOUS_COLS].astype(float) - mean_real) / (std_real + 1e-8)).abs()
    mask = (z > 2).sum(axis=1) > 3
    features = (z[mask] > 2).mean().sort_values(ascending=False)
    return mask, features


mask_ctab, feat_ctab = outlier_analysis(synthetic_ctab)
mask_ctgan, feat_ctgan = outlier_analysis(synthetic_ctgan)

# --- Check: must reproduce the saved notebook output (963 and 91 of 11710) ---
print(f"CTAB-GAN+ outlier records: {mask_ctab.sum()} ({mask_ctab.mean():.2%} of {len(synthetic_ctab)})")
print(f"CTGAN outlier records:     {mask_ctgan.sum()} ({mask_ctgan.mean():.2%} of {len(synthetic_ctgan)})")
if (mask_ctab.sum(), mask_ctgan.sum()) == (963, 91):
    print("OK: identical to the notebook (run C).")
else:
    print("WARNING: different from the notebook (expected 963 and 91). Check the CSVs.")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
panels = [
    (axes[0], "CTAB-GAN+", mask_ctab, feat_ctab, "#4D79FF"),
    (axes[1], "CTGAN", mask_ctgan, feat_ctgan, "#FF4D4D"),
]
for ax, name, mask, feat, color in panels:
    top = feat.head(10)
    ax.barh(top.index[::-1], top.values[::-1], color=color, alpha=0.85)
    ax.axvline(0.5, color="red", linestyle="--", alpha=0.6, label="50% threshold")
    ax.set_title(f"{name} ({mask.mean():.2%} outlier rate, n = {mask.sum()})", fontsize=11)
    ax.set_xlabel("Proportion of outlier records where feature is >2σ")
    ax.legend()

plt.tight_layout()
out = FIG_DIR / "sensitivity_feature_analysis.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved: {out}")


CTAB-GAN+ outlier records: 963 (8.22% of 11710)
CTGAN outlier records:     91 (0.78% of 11710)
OK: identical to the notebook (run C).


Figure saved: ../figures/sensitivity_feature_analysis.png
